# STEP 4: ADR/ODR Label Construction

This notebook builds ADR/ODR labels using:

1. Rule-based labeling for District Court (DDL) data (act/section + criminal/bailable flags).
2. Keyword-based labeling for High Court / Supreme Court text (title + description).

Outputs:
- `ddl_labeled.parquet`
- `hc_labeled.parquet`
- `sc_labeled.parquet`
- `combined_labeled.parquet`
- `training_data.parquet`
- `needs_llm_labeling.parquet`

This version is configured to read parquet files from Google Drive upload locations.

## Setup packages and connect drive


In [1]:
# Colab setup
%pip install -q pandas pyarrow

import pandas as pd
import re
import gc
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Connect Google Drive (Colab)
from google.colab import drive

drive.mount('/content/drive')
print('Connected to the drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to the drive


## Handle DDl inputs and set correct dirs 

In [3]:
# Configuration (Drive paths)
BASE_DIR = Path('/content/drive/MyDrive/MiniProject')
OUTPUT_DIR = BASE_DIR / 'compiled_dataset_180426'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Uploaded dataset folders on Drive
IHCJ_DIR = Path('/content/drive/MyDrive/MiniProject/IHCJ_dataset_180426')
ISCJ_DIR = Path('/content/drive/MyDrive/MiniProject/ISCJ_dataset_180426')

# Chunk size for low-RAM processing (reduce to 25000 if needed)
CHUNK_SIZE = 25000

# Input file candidates (priority order)
DDL_CANDIDATES = [
    OUTPUT_DIR / 'ddl_processed.parquet',
    BASE_DIR / 'compiled_dataset_180426' / 'ddl_processed.parquet',
]
HC_CANDIDATES = [
    IHCJ_DIR / 'hc_metadata.parquet',
    OUTPUT_DIR / 'hc_metadata.parquet',
    BASE_DIR / 'compiled_dataset_180426' / 'hc_metadata.parquet',
]
SC_CANDIDATES = [
    ISCJ_DIR / 'sc_metadata.parquet',
    OUTPUT_DIR / 'sc_metadata.parquet',
    BASE_DIR / 'compiled_dataset_180426' / 'sc_metadata.parquet',
]

# Optional part-file fallback for DDL if single file is unavailable
DDL_PART_SEARCH_DIRS = [
    OUTPUT_DIR,
    BASE_DIR / 'compiled_dataset_180426',
]


def resolve_first_existing(candidates, label):
    for p in candidates:
        if p.exists():
            print(f'{label}: using {p}')
            return p
    print(f'{label}: not found in candidates:')
    for p in candidates:
        print(f'  - {p}')
    return None


def resolve_ddl_input_files():
    single = resolve_first_existing(DDL_CANDIDATES, 'DDL input')
    if single is not None:
        return [single]

    part_files = []
    for d in DDL_PART_SEARCH_DIRS:
        if d.exists():
            part_files.extend(sorted(d.glob('ddl_processed_*_part_*.parquet')))

    if part_files:
        print(f'DDL input: using {len(part_files)} part files')
        return part_files

    print('DDL input: no single file or part files found')
    return []

## Label rules

In [4]:
# Label rules
ADR_ELIGIBLE_ACTS = {
    'motor vehicles':       (1, 1, 'Lok Adalat / MACT - high volume, routine, ODR-friendly'),
    'consumer protection':  (1, 1, 'Consumer forum / Lok Adalat - often small claims, ODR-friendly'),
    'arbitration':          (1, 0, 'Already in arbitration - ADR yes, ODR depends on complexity'),
    'negotiable instruments': (1, 1, 'Cheque bounce - compoundable, Lok Adalat eligible, ODR-friendly'),
    'commercial courts':    (1, 0, 'Pre-institution mediation mandatory under Commercial Courts Act'),
    'specific relief':      (1, 0, 'Civil contract disputes - CPC S.89 ADR referral'),
    'transfer of property': (1, 0, 'Property/rent disputes - mediation eligible'),
    'contract':             (1, 0, 'Contract disputes - core arbitration territory'),
    'insurance':            (1, 1, 'Insurance claims - ODR-friendly if quantum is main issue'),
    'recovery of debts':    (1, 0, 'Debt recovery - mediation / Lok Adalat eligible'),
    'micro, small':         (1, 0, 'MSME Act - mandatory conciliation'),
    'electricity':          (1, 0, 'Electricity disputes - ombudsman / ADR'),
    'real estate':          (1, 0, 'RERA - conciliation eligible'),
    'hindu marriage':       (1, 0, 'Family matters - court-annexed mediation, not ODR'),
    'family courts':        (1, 0, 'Family matters - mediation eligible'),
    'special marriage':     (1, 0, 'Family matters - mediation eligible'),
    'maintenance':          (1, 0, 'Family/maintenance - mediation eligible'),
    'partnership':          (1, 0, 'Partnership disputes - arbitration eligible'),
    'companies act':        (1, 0, 'Company disputes - NCLT conciliation'),
    'insolvency':           (1, 0, 'IBC - settlement possible at pre-admission stage'),
    'labour':               (1, 0, 'Labour disputes - conciliation under ID Act'),
    'industrial disputes':  (1, 0, 'Conciliation mandatory under Industrial Disputes Act'),
    'workmen':              (1, 0, 'Labour - conciliation eligible'),
    'employees':            (1, 0, 'Labour - conciliation eligible'),
    'land acquisition':     (1, 0, 'Compensation disputes - Lok Adalat eligible'),
    'public premises':      (1, 0, 'Eviction - mediation eligible'),
    'rent control':         (1, 0, 'Rent disputes - mediation eligible'),

    'indian penal code':    (0, 0, 'IPC - criminal, non-compoundable offences not ADR eligible'),
    'ipc':                  (0, 0, 'IPC criminal'),
    'prevention of corruption': (0, 0, 'Anti-corruption - public interest, not ADR'),
    'narcotic':             (0, 0, 'NDPS - criminal, non-compoundable'),
    'pocso':                (0, 0, 'Child protection - not ADR eligible'),
    'protection of children': (0, 0, 'Child protection - not ADR eligible'),
    'terrorism':            (0, 0, 'UAPA - not ADR eligible'),
    'unlawful activities':  (0, 0, 'UAPA - not ADR eligible'),
    'arms act':             (0, 0, 'Criminal - not ADR eligible'),
    'explosives':           (0, 0, 'Criminal - not ADR eligible'),
    'scheduled castes':     (0, 0, 'SC/ST Act - not ADR eligible (constitutional protection)'),
    'atrocities':           (0, 0, 'SC/ST Act - not ADR eligible'),
    'constitution':         (0, 0, 'Constitutional matters - not ADR eligible'),
    'fundamental rights':   (0, 0, 'Writ jurisdiction - not ADR eligible'),
    'habeas corpus':        (0, 0, 'Not ADR eligible'),
    'election':             (0, 0, 'Election disputes - statutory process'),
    'contempt':             (0, 0, 'Contempt - not ADR eligible'),
    'revenue':              (0, 0, 'Revenue/tax - usually not ADR eligible'),
    'income tax':           (0, 0, 'Tax - not ADR eligible'),
    'customs':              (0, 0, 'Tax/customs - not ADR eligible'),
    'foreign exchange':     (0, 0, 'FEMA - not ADR eligible'),
}

BAILABLE_LABEL = (1, 0)
NON_BAILABLE_LABEL = (0, 0)

ADR_POSITIVE_KEYWORDS = [
    'arbitration', 'mediation', 'conciliation', 'lok adalat',
    'settlement agreement', 'section 89 cpc', 'adr', 'odr',
    'online dispute resolution', 'negotiated settlement',
    'arbitral award', 'arbitral tribunal', 'mediator',
    'mutual consent', 'amicable settlement',
    'pre-litigation', 'pre institution mediation',
]
NON_ADR_KEYWORDS = [
    'murder', 'rape', 'dacoity', 'terrorism', 'pocso',
    'habeas corpus', 'writ petition', 'fundamental right',
    'election', 'contempt of court', 'ndps', 'narcotic',
]
ODR_POSITIVE_KEYWORDS = [
    'online', 'digital', 'e-commerce', 'electronic', 'cyber',
    'internet', 'odr', 'online dispute', 'cheque bounce',
    'motor accident', 'consumer complaint', 'insurance claim',
]
NEGATION_TERMS = [
    'no', 'not', 'without', 'reject', 'rejected', 'rejects', 'rejecting',
    'deny', 'denied', 'denies', 'denying', 'decline', 'declined',
    'declines', 'declining', 'refuse', 'refused', 'refuses', 'refusing',
    'inapplicable', 'barred', 'impermissible',
]

In [5]:
def label_from_act(act_s: str, section_s: str = None, criminal: float = None, bailable: float = None) -> tuple:
    act = str(act_s).lower() if pd.notna(act_s) else ''
    section = str(section_s).lower() if pd.notna(section_s) else ''

    for key, (adr, odr, reason) in ADR_ELIGIBLE_ACTS.items():
        if key in act or key in section:
            return (adr, odr, reason)

    if pd.notna(criminal):
        if int(criminal) == 0:
            return (1, 0, 'Civil case - CPC S.89 ADR referral applicable')
        else:
            if pd.notna(bailable) and int(bailable) == 1:
                return BAILABLE_LABEL[0], BAILABLE_LABEL[1], 'Bailable criminal - Lok Adalat eligible'
            return NON_BAILABLE_LABEL[0], NON_BAILABLE_LABEL[1], 'Non-bailable criminal - not ADR eligible'

    return (-1, -1, 'Unknown - needs manual review')


def label_from_text(title: str = None, description: str = None) -> tuple:
    text = ' '.join(filter(None, [
        str(title).lower() if pd.notna(title) else '',
        str(description).lower() if pd.notna(description) else '',
    ]))

    if not text.strip():
        return (-1, -1, 'No text available')

    def keyword_in_text(keyword: str) -> bool:
        pattern = r'(?<!\\w)' + re.escape(keyword).replace(r'\\ ', r'\\s+') + r'(?!\\w)'
        return re.search(pattern, text) is not None

    def negated_keyword(keyword: str) -> bool:
        key = re.escape(keyword).replace(r'\\ ', r'\\s+')
        neg = r'(?:' + '|'.join(re.escape(t) for t in NEGATION_TERMS) + r')'
        before = rf'{neg}(?:\\W+\\w+){{0,4}}\\W+{key}'
        after = rf'{key}(?:\\W+\\w+){{0,4}}\\W+{neg}'
        return re.search(before, text) is not None or re.search(after, text) is not None

    negative_hits = [kw for kw in NON_ADR_KEYWORDS if keyword_in_text(kw)]
    adr_hits = [kw for kw in ADR_POSITIVE_KEYWORDS if keyword_in_text(kw)]
    odr_hits = [kw for kw in ODR_POSITIVE_KEYWORDS if keyword_in_text(kw)]
    negated_adr_hits = [kw for kw in adr_hits if negated_keyword(kw)]
    negated_odr_hits = [kw for kw in odr_hits if negated_keyword(kw)]

    if negative_hits and (adr_hits or odr_hits):
        return (-1, -1, 'Ambiguous text: mixed ADR and non-ADR signals')
    if negated_adr_hits or negated_odr_hits:
        return (-1, -1, 'Ambiguous text: ADR/ODR terms appear in negated context')
    if negative_hits:
        return (0, 0, f"Non-ADR keyword found: '{negative_hits[0]}'")
    if not adr_hits and not odr_hits:
        return (-1, -1, 'No ADR/ODR signal in text')

    adr = 1 if (adr_hits or odr_hits) else 0
    odr = 1 if odr_hits else 0
    reasons = []
    if adr_hits:
        reasons.append(f"ADR keyword: '{adr_hits[0]}'")
    if odr_hits:
        reasons.append(f"ODR keyword: '{odr_hits[0]}'")

    return (adr, odr, '; '.join(reasons))


def label_ddl(df: pd.DataFrame) -> pd.DataFrame:
    print(f'  Labeling {len(df):,} DDL rows ...')

    results = df.apply(
        lambda r: label_from_act(r.get('act_s'), r.get('section_s'), r.get('criminal'), r.get('bailable_ipc')),
        axis=1,
        result_type='expand'
    )
    results.columns = ['adr_label', 'odr_label', 'label_reason']
    df = pd.concat([df, results], axis=1)

    df['final_label'] = df.apply(
        lambda r: -1 if r['adr_label'] == -1 else (2 if r['odr_label'] == 1 else r['adr_label']),
        axis=1
    )
    df['source'] = 'DDL_district_court'
    return df


def label_court_text(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    print(f'  Labeling {len(df):,} {source_name} rows ...')

    title_col = 'title' if 'title' in df.columns else None
    desc_col = 'description' if 'description' in df.columns else None

    results = df.apply(
        lambda r: label_from_text(r.get(title_col) if title_col else None, r.get(desc_col) if desc_col else None),
        axis=1,
        result_type='expand'
    )
    results.columns = ['adr_label', 'odr_label', 'label_reason']
    df = pd.concat([df, results], axis=1)

    df['final_label'] = df.apply(
        lambda r: -1 if r['adr_label'] == -1 else (2 if r['odr_label'] == 1 else r['adr_label']),
        axis=1
    )
    df['source'] = source_name
    return df

In [6]:
UNIFIED_COLS = [
    'source', 'case_id', 'court_level', 'court_name', 'year',
    'case_type', 'act', 'section', 'title', 'description',
    'decision_date', 'date_of_filing', 'disposal_nature',
    'is_criminal', 'is_bailable', 'state',
    'adr_label', 'odr_label', 'adr_target', 'odr_target',
    'final_label', 'label_reason',
]


def standardise(df: pd.DataFrame, court_level: str) -> pd.DataFrame:
    out = pd.DataFrame()
    out['source'] = df.get('source', pd.Series([court_level] * len(df)))
    out['court_level'] = court_level

    for id_col in ['ddl_case_id', 'cnr', 'case_id']:
        if id_col in df.columns:
            out['case_id'] = df[id_col].astype(str)
            break
    else:
        out['case_id'] = range(len(df))

    for col in ['court_name_clean', 'court_name', 'state_name']:
        if col in df.columns:
            out['court_name'] = df[col]
            break

    out['year'] = df.get('source_year', df.get('year', pd.Series([None] * len(df))))

    mapping = [
        ('type_name_s', 'case_type'), ('act_s', 'act'), ('section_s', 'section'),
        ('title', 'title'), ('description', 'description'),
        ('decision_date', 'decision_date'), ('date_of_filing', 'date_of_filing'),
        ('disp_name_s', 'disposal_nature'), ('disposal_nature', 'disposal_nature'),
        ('criminal', 'is_criminal'), ('bailable_ipc', 'is_bailable'), ('state_name', 'state'),
    ]

    for src, tgt in mapping:
        if src in df.columns and tgt not in out.columns:
            out[tgt] = df[src]

    for col in ['adr_label', 'odr_label', 'final_label', 'label_reason']:
        out[col] = df[col]

    out['adr_target'] = out['adr_label']
    out['odr_target'] = out['odr_label']

    for col in UNIFIED_COLS:
        if col not in out.columns:
            out[col] = pd.NA
    out = out[UNIFIED_COLS].copy()

    # Cast to consistent types to ensure schema uniformity
    for col in ['source', 'case_id', 'court_level', 'court_name', 'case_type', 'act', 'section', 'title', 'description', 'disposal_nature', 'state', 'label_reason']:
        out[col] = out[col].astype('string')
    for col in ['adr_label', 'odr_label', 'adr_target', 'odr_target', 'final_label', 'year', 'is_criminal', 'is_bailable']:
        out[col] = pd.to_numeric(out[col], errors='coerce').astype('Int64')
    for col in ['decision_date', 'date_of_filing']:
        out[col] = pd.to_datetime(out[col], errors='coerce')

    return out


def validate_standardised(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    missing = [c for c in UNIFIED_COLS if c not in df.columns]
    assert not missing, f"{source_name}: missing columns: {missing}"

    allowed_final = {-1, 0, 1, 2}
    vals = set(df['final_label'].dropna().astype(int).unique().tolist())
    assert vals.issubset(allowed_final), f"{source_name}: invalid final_label values: {sorted(vals - allowed_final)}"

    allowed_bin = {-1, 0, 1}
    for col in ['adr_label', 'odr_label', 'adr_target', 'odr_target']:
        vals = set(df[col].dropna().astype(int).unique().tolist())
        assert vals.issubset(allowed_bin), f"{source_name}: invalid {col} values: {sorted(vals - allowed_bin)}"

    dup_count = int(df.duplicated(subset=['source', 'case_id']).sum())
    if dup_count > 0:
        print(f"Warning: {source_name}: removing {dup_count} duplicate (source, case_id) rows")
        df = df.drop_duplicates(subset=['source', 'case_id'])

    inconsistent_odr = df[(df['final_label'] == 2) & (df['odr_label'] != 1)]
    assert inconsistent_odr.empty, f"{source_name}: final_label=2 requires odr_label=1"

    inconsistent_adr = df[(df['final_label'] == 1) & (df['adr_label'] != 1)]
    assert inconsistent_adr.empty, f"{source_name}: final_label=1 requires adr_label=1"

    inconsistent_unknown = df[(df['final_label'] == -1) & ((df['adr_label'] != -1) | (df['odr_label'] != -1))]
    assert inconsistent_unknown.empty, f"{source_name}: final_label=-1 requires adr_label=odr_label=-1"

    return df

In [7]:
# Step 1 checkpoint / resume configuration
STEP1_CHECKPOINT = OUTPUT_DIR / 'ddl_labeled_step1_checkpoint.parquet'
FORCE_RERUN_STEP1 = True  # Set to True only if you want to rebuild Step 1 labels

print(f'STEP1_CHECKPOINT: {STEP1_CHECKPOINT}')
print(f'FORCE_RERUN_STEP1: {FORCE_RERUN_STEP1}')

STEP1_CHECKPOINT: /content/drive/MyDrive/MiniProject/compiled_dataset_180426/ddl_labeled_step1_checkpoint.parquet
FORCE_RERUN_STEP1: True


In [8]:
# Run Step 4 (low-RAM chunked pipeline)
import shutil

print('=' * 60)
print('ADR/ODR Label Constructor (Low-RAM Mode)')
print('=' * 60)
print(f'CHUNK_SIZE={CHUNK_SIZE:,}')

combined_path = OUTPUT_DIR / 'combined_labeled.parquet'
training_path = OUTPUT_DIR / 'training_data.parquet'
unknown_path = OUTPUT_DIR / 'needs_llm_labeling.parquet'

ddl_out = OUTPUT_DIR / 'ddl_labeled.parquet'
hc_out = OUTPUT_DIR / 'hc_labeled.parquet'
sc_out = OUTPUT_DIR / 'sc_labeled.parquet'

# Fallback defaults if the checkpoint config cell was not run
if 'STEP1_CHECKPOINT' not in globals():
    STEP1_CHECKPOINT = OUTPUT_DIR / 'ddl_labeled_step1_checkpoint.parquet'
if 'FORCE_RERUN_STEP1' not in globals():
    FORCE_RERUN_STEP1 = False

print(f'Step 1 checkpoint path: {STEP1_CHECKPOINT}')
print(f'Force rerun Step 1: {FORCE_RERUN_STEP1}')

# Reset outputs for a fresh run (keep Step 1 checkpoint file)
for p in [combined_path, training_path, unknown_path, ddl_out, hc_out, sc_out]:
    if p.exists():
        p.unlink()

rows_total = 0
rows_known = 0
rows_unknown = 0
label_counts = {0: 0, 1: 0, 2: 0}


class StreamingParquetWriter:
    def __init__(self, out_path: Path):
        self.out_path = out_path
        self.writer = None

    def write_df(self, df: pd.DataFrame):
        if df is None or df.empty:
            return
        table = pa.Table.from_pandas(df, preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(self.out_path, table.schema)
        self.writer.write_table(table)

    def close(self):
        if self.writer is not None:
            self.writer.close()
            self.writer = None


w_combined = StreamingParquetWriter(combined_path)
w_training = StreamingParquetWriter(training_path)
w_unknown = StreamingParquetWriter(unknown_path)
w_ddl = StreamingParquetWriter(ddl_out)
w_hc = StreamingParquetWriter(hc_out)
w_sc = StreamingParquetWriter(sc_out)


try:
    def write_labeled_outputs(std: pd.DataFrame, source_writer: StreamingParquetWriter):
        source_writer.write_df(std)
        w_combined.write_df(std)

        known = std[std['final_label'] != -1].copy()
        unknown = std[std['final_label'] == -1].copy()
        w_training.write_df(known)
        w_unknown.write_df(unknown)

        nonlocal_rows = len(std)
        nonlocal_known = len(known)
        nonlocal_unknown = len(unknown)

        globals()['rows_total'] += nonlocal_rows
        globals()['rows_known'] += nonlocal_known
        globals()['rows_unknown'] += nonlocal_unknown

        vc = known['final_label'].value_counts(dropna=True).to_dict()
        for k in [0, 1, 2]:
            label_counts[k] += int(vc.get(k, 0))

        del known, unknown
        return nonlocal_rows


    def process_df_in_chunks(df_src: pd.DataFrame, label_fn, source_name: str, court_level: str, source_writer: StreamingParquetWriter):
        local_total = 0

        if df_src.empty:
            print(f'  {source_name}: empty input, skipped')
            return 0

        for start in range(0, len(df_src), CHUNK_SIZE):
            chunk = df_src.iloc[start:start + CHUNK_SIZE].copy()

            labeled = label_fn(chunk)
            std = standardise(labeled, court_level)
            std = validate_standardised(std, source_name)

            local_total += write_labeled_outputs(std, source_writer)

            del chunk, labeled, std
            gc.collect()

        return local_total


    def process_parquet_in_batches(parquet_path: Path, label_fn, source_name: str, court_level: str, source_writer: StreamingParquetWriter):
        local_total = 0
        pf = pq.ParquetFile(parquet_path)
        approx_rows = pf.metadata.num_rows if pf.metadata is not None else 0
        print(f'  {source_name}: streaming {approx_rows:,} rows in batches of {CHUNK_SIZE:,}')

        batch_count = 0
        for batch_count, batch in enumerate(pf.iter_batches(batch_size=CHUNK_SIZE), start=1):
            chunk = batch.to_pandas()
            if chunk.empty:
                continue

            # Drop duplicates to avoid validation errors
            chunk = chunk.drop_duplicates()

            labeled = label_fn(chunk)
            std = standardise(labeled, court_level)
            std = validate_standardised(std, source_name)

            local_total += write_labeled_outputs(std, source_writer)

            if batch_count % 20 == 0:
                print(f'    {source_name}: processed {batch_count:,} batches')

            del chunk, labeled, std, batch
            gc.collect()

        print(f'  {source_name}: processed {batch_count:,} batches total')
        return local_total


    # [1] DDL
    print('\n[1] Labeling DDL District Court data ...')
    use_step1_checkpoint = STEP1_CHECKPOINT.exists() and (not FORCE_RERUN_STEP1)

    # Optimized version (stream in chunks):
    if use_step1_checkpoint:
        print(f'    Streaming Step 1 checkpoint: {STEP1_CHECKPOINT}')
        pf = pq.ParquetFile(STEP1_CHECKPOINT)
        approx_rows = pf.metadata.num_rows if pf.metadata is not None else 0
        print(f'    Checkpoint has ~{approx_rows:,} rows; processing in batches of {CHUNK_SIZE:,}')
        
        local_total = 0
        batch_count = 0
        for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
            chunk = batch.to_pandas()
            if chunk.empty:
                continue
            
            # Re-standardize to ensure consistent schema
            std_chunk = standardise(chunk, 'District Court')
            std_chunk = validate_standardised(std_chunk, 'District Court')
            
            local_total += write_labeled_outputs(std_chunk, w_ddl)
            batch_count += 1
            
            if batch_count % 10 == 0:
                print(f'    Processed {batch_count} checkpoint batches')
                gc.collect()  # Aggressive GC
            
            del chunk, std_chunk, batch
            gc.collect()
        
        w_ddl.close()
        print(f'    Rebuilt ddl_labeled.parquet from checkpoint ({local_total:,} rows in {batch_count} batches)')
        del pf
        gc.collect()


    # [2] High Court
    print('\n[2] Labeling High Court data ...')
    hc_path = resolve_first_existing(HC_CANDIDATES, 'High Court input')
    if hc_path is not None:
        total_local = process_parquet_in_batches(
            hc_path,
            lambda x: label_court_text(x, 'High Court'),
            'High Court',
            'High Court',
            w_hc
        )
        print(f'    Saved hc_labeled.parquet ({total_local:,} rows)')
        gc.collect()
    else:
        print('    SKIP - High Court parquet not found')


    # [3] Supreme Court
    print('\n[3] Labeling Supreme Court data ...')
    sc_path = resolve_first_existing(SC_CANDIDATES, 'Supreme Court input')
    if sc_path is not None:
        total_local = process_parquet_in_batches(
            sc_path,
            lambda x: label_court_text(x, 'Supreme Court'),
            'Supreme Court',
            'Supreme Court',
            w_sc
        )
        print(f'    Saved sc_labeled.parquet ({total_local:,} rows)')
        gc.collect()
    else:
        print('    SKIP - Supreme Court parquet not found')

finally:
    w_combined.close()
    w_training.close()
    w_unknown.close()
    w_ddl.close()
    w_hc.close()
    w_sc.close()


# [4] Summary
print('\n[4] Final Summary ...')
print(f'    Total rows:   {rows_total:,}')
print(f'    Labeled rows: {rows_known:,}')
print(f'    Unknown rows: {rows_unknown:,} (optional Step 5 with Gemini)')

print('\n    Label distribution (known rows):')
label_map = {0: 'NOT eligible', 1: 'ADR eligible', 2: 'ADR + ODR eligible'}
for k in [0, 1, 2]:
    print(f"    {label_map[k]}: {label_counts[k]:,}")

print('\n    Saved combined_labeled.parquet')
print('    Saved training_data.parquet')
print('    Saved needs_llm_labeling.parquet')
print('\nDone')

ADR/ODR Label Constructor (Low-RAM Mode)
CHUNK_SIZE=25,000
Step 1 checkpoint path: /content/drive/MyDrive/MiniProject/compiled_dataset_180426/ddl_labeled_step1_checkpoint.parquet
Force rerun Step 1: True

[1] Labeling DDL District Court data ...

[2] Labeling High Court data ...
High Court input: using /content/drive/MyDrive/MiniProject/IHCJ_dataset_180426/hc_metadata.parquet
  High Court: streaming 3,433,684 rows in batches of 25,000
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court rows ...
  Labeling 25,000 High Court 